# Examen ML — Prédiction des matchs de la Coupe du Monde

**Sources :**
- `results.csv` : historique des matchs internationaux (1872 → 2026), y compris les matchs de Coupe du Monde
- `fifa_ranking-2024-06-20.csv` : classement FIFA historique (1992 → juin 2024)

**Plan (suit l'énoncé) :**
1. Feature engineering de la base historique des matchs (CM)
2. Feature engineering du classement FIFA
3. Fusion des deux bases (clé : équipe + date la plus proche du classement)
4. Construction de la variable cible
5. Entraînement / comparaison de 3 modèles ML + plateforme de prédiction

**Remarque sur les données :** le fichier `results.csv` contient déjà les matchs de la Coupe du Monde 2026 jusqu'au 6 juillet 2026, avec **9 matchs sans score** (quarts de finale → finale). Ce sont précisément les matchs que notre plateforme doit prédire.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

results = pd.read_csv('/mnt/user-data/uploads/results__1_.csv')
ranking = pd.read_csv('/mnt/user-data/uploads/fifa_ranking-2024-06-20.csv')

print(results.shape, ranking.shape)
results.head()

(49499, 9) (67472, 8)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## Étape 1 — Feature engineering de la base historique (results.csv)

Sous-étapes :
- harmonisation des noms de pays (clé de jointure commune avec le classement FIFA)
- construction du résultat du match (H/D/A), de l'écart de buts
- séparation des matchs déjà joués / des 9 matchs à prédire (score manquant)
- calcul de features de **forme d'équipe** glissante (5 et 10 derniers matchs), calculées de façon **causale** (uniquement à partir des matchs strictement antérieurs) pour éviter toute fuite de données (data leakage)


In [2]:
NAME_MAP_RESULTS_TO_RANKING = {
    'Cape Verde': 'Cabo Verde',
    'China': 'China PR',
    'Curaçao': 'Curacao',
    'Czech Republic': 'Czechia',
    'DR Congo': 'Congo DR',
    # 'German DR' (ex-RDA, dissoute en 1990) : aucune entrée dans le classement
    # FIFA (qui démarre en 1992) -> laissé tel quel, restera NaN (cohérent).
    'Iran': 'IR Iran',
    'Ivory Coast': "Côte d'Ivoire",
    'North Korea': 'Korea DPR',
    'South Korea': 'Korea Republic',
    'United States': 'USA',
}

results['home_team_std'] = results['home_team'].replace(NAME_MAP_RESULTS_TO_RANKING)
results['away_team_std'] = results['away_team'].replace(NAME_MAP_RESULTS_TO_RANKING)

results['date'] = pd.to_datetime(results['date'])
results = results.sort_values('date').reset_index(drop=True)
results['match_id'] = results.index

to_predict = results[results['home_score'].isna()].copy()
played = results[results['home_score'].notna()].copy()
played['home_score'] = played['home_score'].astype(int)
played['away_score'] = played['away_score'].astype(int)

print('Matchs joués :', played.shape[0])
print('Matchs à prédire (score manquant) :', to_predict.shape[0])
to_predict[['date', 'home_team', 'away_team', 'tournament']]

Matchs joués : 49490
Matchs à prédire (score manquant) : 9


,date,home_team,away_team,tournament
49490,2026-07-03,Australia,Egypt,FIFA World Cup
49491,2026-07-03,Argentina,Cape Verde,FIFA World Cup
49492,2026-07-03,Colombia,Ghana,FIFA World Cup
49493,2026-07-04,Canada,Morocco,FIFA World Cup
49494,2026-07-04,Paraguay,France,FIFA World Cup
49495,2026-07-05,Brazil,Norway,FIFA World Cup
49496,2026-07-05,Mexico,England,FIFA World Cup
49497,2026-07-06,Portugal,Spain,FIFA World Cup
49498,2026-07-06,United States,Belgium,FIFA World Cup


In [3]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'H'
    elif row['home_score'] < row['away_score']:
        return 'A'
    return 'D'

played['result'] = played.apply(get_result, axis=1)
played['goal_diff'] = played['home_score'] - played['away_score']
played['total_goals'] = played['home_score'] + played['away_score']
played['year'] = played['date'].dt.year
played['month'] = played['date'].dt.month
played['is_world_cup'] = (played['tournament'] == 'FIFA World Cup').astype(int)
played['is_qualifier'] = played['tournament'].str.contains('qualification', case=False, na=False).astype(int)
played['is_friendly'] = (played['tournament'] == 'Friendly').astype(int)
played['is_neutral'] = played['neutral'].astype(bool).astype(int)

played[['date', 'home_team', 'away_team', 'result', 'goal_diff', 'is_world_cup', 'is_neutral']].head()

,date,home_team,away_team,result,goal_diff,is_world_cup,is_neutral
0,1872-11-30,Scotland,England,D,0,0,0
1,1873-03-08,England,Scotland,H,2,0,0
2,1874-03-07,Scotland,England,H,1,0,0
3,1875-03-06,England,Scotland,D,0,0,0
4,1876-03-04,Scotland,England,H,3,0,0


### Features de forme d'équipe (rolling, causales)

On reconstruit un historique "long format" (1 ligne par équipe et par match) trié chronologiquement, puis on calcule des statistiques glissantes avec `shift(1)` pour n'utiliser que le passé de chaque équipe.


In [4]:
home_side = played[['match_id', 'date', 'home_team_std', 'away_team_std', 'home_score', 'away_score', 'is_neutral']].copy()
home_side.columns = ['match_id', 'date', 'team', 'opponent', 'gf', 'ga', 'is_neutral']
home_side['is_home'] = 1

away_side = played[['match_id', 'date', 'away_team_std', 'home_team_std', 'away_score', 'home_score', 'is_neutral']].copy()
away_side.columns = ['match_id', 'date', 'team', 'opponent', 'gf', 'ga', 'is_neutral']
away_side['is_home'] = 0

long_df = pd.concat([home_side, away_side], ignore_index=True).sort_values(['team', 'date']).reset_index(drop=True)

long_df['points'] = np.select([long_df['gf'] > long_df['ga'], long_df['gf'] == long_df['ga']], [3, 1], default=0)
long_df['win'] = (long_df['gf'] > long_df['ga']).astype(int)

grp = long_df.groupby('team', group_keys=False)
long_df['form5_winrate'] = grp['win'].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
long_df['form5_avg_gf'] = grp['gf'].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
long_df['form5_avg_ga'] = grp['ga'].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
long_df['form10_winrate'] = grp['win'].apply(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
long_df['career_matches'] = grp.cumcount()
long_df['career_winrate'] = grp['win'].apply(lambda s: s.shift(1).expanding(min_periods=1).mean())
long_df['career_avg_points'] = grp['points'].apply(lambda s: s.shift(1).expanding(min_periods=1).mean())

fill_cols = ['form5_winrate', 'form5_avg_gf', 'form5_avg_ga', 'form10_winrate', 'career_winrate', 'career_avg_points']
for c in fill_cols:
    long_df[c] = long_df[c].fillna(long_df[c].median())

long_df.head()

,match_id,date,team,opponent,gf,ga,is_neutral,is_home,points,win,form5_winrate,form5_avg_gf,form5_avg_ga,form10_winrate,career_matches,career_winrate,career_avg_points
0,36260,2012-09-25,Abkhazia,Artsakh,1,1,0,1,1,0,0.400000,1.400000,1.200000,0.400000,0,0.380567,1.365385
1,36404,2012-10-21,Abkhazia,Artsakh,0,3,0,0,0,0,0.000000,1.000000,1.000000,0.000000,1,0.000000,1.000000
2,37317,2013-09-23,Abkhazia,South Ossetia,3,0,0,1,3,1,0.000000,0.500000,2.000000,0.000000,2,0.000000,0.500000
3,37780,2014-06-01,Abkhazia,Occitania,1,1,1,1,1,0,0.333333,1.333333,1.333333,0.333333,3,0.333333,1.333333
4,37798,2014-06-02,Abkhazia,Sápmi,2,1,0,0,3,1,0.250000,1.250000,1.250000,0.250000,4,0.250000,1.250000


## Étape 2 — Feature engineering du classement FIFA

- `points_change` : évolution des points depuis la publication précédente
- `rank_improved` : progression du rang (signe positif = amélioration)
- `rank_trend3` / `points_trend3` : tendance sur les 3 dernières publications (dynamique récente de l'équipe)


In [5]:
ranking['rank_date'] = pd.to_datetime(ranking['rank_date'])
ranking = ranking.sort_values(['country_full', 'rank_date']).reset_index(drop=True)

ranking['points_change'] = ranking['total_points'] - ranking['previous_points']
ranking['rank_improved'] = -ranking['rank_change']

g = ranking.groupby('country_full', group_keys=False)
ranking['rank_trend3'] = g['rank'].apply(lambda s: s.diff().rolling(3, min_periods=1).mean())
ranking['points_trend3'] = g['points_change'].apply(lambda s: s.rolling(3, min_periods=1).mean())
ranking[['rank_trend3', 'points_trend3']] = ranking[['rank_trend3', 'points_trend3']].fillna(0)

ranking[['country_full', 'rank_date', 'rank', 'total_points', 'points_change', 'rank_improved', 'rank_trend3']].head()

,country_full,rank_date,rank,total_points,points_change,rank_improved,rank_trend3
0,Afghanistan,2003-01-15,204.0,7.0,7.0,-204,0.000000
1,Afghanistan,2003-02-19,203.0,9.0,2.0,1,-1.000000
2,Afghanistan,2003-03-26,198.0,48.0,39.0,5,-3.000000
3,Afghanistan,2003-04-23,198.0,48.0,0.0,0,-2.000000
4,Afghanistan,2003-05-21,199.0,48.0,0.0,-1,-1.333333


## Étape 3 — Fusion des deux bases

**Clé de jointure : équipe (nom standardisé) + date du classement FIFA la plus proche (≤) de la date du match**, réalisée avec `pd.merge_asof` (direction `backward`), séparément pour l'équipe à domicile et l'équipe à l'extérieur.


In [6]:
RANK_COLS = ['rank', 'total_points', 'previous_points', 'points_change', 'rank_improved', 'rank_trend3', 'points_trend3', 'confederation']

def merge_asof_ranking(df, team_std_col, prefix):
    left = df[['match_id', 'date', team_std_col]].rename(columns={team_std_col: 'country_full'}).sort_values('date')
    right = ranking[['country_full', 'rank_date'] + RANK_COLS].sort_values('rank_date')
    merged = pd.merge_asof(left, right, left_on='date', right_on='rank_date', by='country_full', direction='backward')
    merged = merged.rename(columns={c: f'{prefix}_{c}' for c in RANK_COLS + ['rank_date']})
    return merged[['match_id'] + [f'{prefix}_{c}' for c in RANK_COLS + ['rank_date']]]

def merge_asof_form(df, team_std_col, prefix):
    left = df[['match_id', team_std_col]].rename(columns={team_std_col: 'team'})
    form_cols = ['form5_winrate', 'form5_avg_gf', 'form5_avg_ga', 'form10_winrate', 'career_matches', 'career_winrate', 'career_avg_points']
    right = long_df[['match_id', 'team'] + form_cols]
    merged = left.merge(right, on=['match_id', 'team'], how='left')
    return merged.rename(columns={c: f'{prefix}_{c}' for c in form_cols})[['match_id'] + [f'{prefix}_{c}' for c in form_cols]]

def build_dataset(df):
    df = df.copy()
    out = df.merge(merge_asof_ranking(df, 'home_team_std', 'home'), on='match_id', how='left') \
            .merge(merge_asof_ranking(df, 'away_team_std', 'away'), on='match_id', how='left') \
            .merge(merge_asof_form(df, 'home_team_std', 'home'), on='match_id', how='left') \
            .merge(merge_asof_form(df, 'away_team_std', 'away'), on='match_id', how='left')
    return out

played_full = build_dataset(played)
to_predict_full = build_dataset(to_predict)

print('Shape matchs joués fusionnés :', played_full.shape)
print('Shape matchs à prédire fusionnés :', to_predict_full.shape)
played_full[['date','home_team','away_team','home_rank','away_rank','home_total_points','away_total_points']].tail()

Shape matchs joués fusionnés : (49490, 53)
Shape matchs à prédire fusionnés : (9, 44)


,date,home_team,away_team,home_rank,away_rank,home_total_points,away_total_points
49485,2026-07-01,Belgium,Senegal,3.0,18.0,1797.98,1623.34
49486,2026-07-01,United States,Bosnia and Herzegovina,11.0,75.0,1676.52,1332.30
49487,2026-07-02,Spain,Austria,8.0,25.0,1729.92,1560.03
49488,2026-07-02,Portugal,Croatia,6.0,9.0,1747.04,1728.30
49489,2026-07-02,Switzerland,Algeria,19.0,44.0,1617.24,1474.13


Le classement FIFA ne démarre qu'en décembre 1992 : les matchs antérieurs auront un classement manquant (`NaN`) et seront exclus de l'entraînement (logique, cette information n'existait pas).

## Étape 4 — Construction de la variable cible

Variable cible **multi-classe (3 classes)**, du point de vue de l'équipe à domicile :
- `0` = victoire extérieure (Away win)
- `1` = match nul (Draw)
- `2` = victoire à domicile (Home win)

On construit ensuite des **features différentielles** (domicile − extérieur), qui donnent au modèle une lecture directe du rapport de force entre les deux équipes.


In [7]:
df = played_full.dropna(subset=['home_rank', 'away_rank']).copy()
print('Matchs exploitables (classement FIFA connu) :', df.shape[0])
print('Période :', df['date'].min().date(), '->', df['date'].max().date())

target_map = {'A': 0, 'D': 1, 'H': 2}
df['target'] = df['result'].map(target_map)

df['target'].value_counts(normalize=True).rename({0: 'Away win', 1: 'Draw', 2: 'Home win'}).round(3)

Matchs exploitables (classement FIFA connu) : 27423
Période : 1993-01-01 -> 2026-07-02


target
Home win    0.482
Away win    0.278
Draw        0.240
Name: proportion, dtype: float64

In [8]:
df['rank_diff'] = df['away_rank'] - df['home_rank']
df['points_diff'] = df['home_total_points'] - df['away_total_points']
df['rank_trend_diff'] = df['home_rank_trend3'] - df['away_rank_trend3']
df['form5_winrate_diff'] = df['home_form5_winrate'] - df['away_form5_winrate']
df['form10_winrate_diff'] = df['home_form10_winrate'] - df['away_form10_winrate']
df['goal_avg_diff'] = (df['home_form5_avg_gf'] - df['home_form5_avg_ga']) - (df['away_form5_avg_gf'] - df['away_form5_avg_ga'])
df['career_winrate_diff'] = df['home_career_winrate'] - df['away_career_winrate']
df['experience_diff'] = df['home_career_matches'] - df['away_career_matches']

FEATURES = [
    'rank_diff', 'points_diff', 'rank_trend_diff', 'form5_winrate_diff', 'form10_winrate_diff',
    'goal_avg_diff', 'career_winrate_diff', 'experience_diff',
    'home_rank', 'away_rank', 'home_total_points', 'away_total_points',
    'home_form5_winrate', 'away_form5_winrate',
    'is_neutral', 'is_world_cup', 'is_qualifier', 'is_friendly',
]
X = df[FEATURES].fillna(df[FEATURES].median())
y = df['target']
X.describe().T[['mean', 'std', 'min', 'max']]

,mean,std,min,max
rank_diff,3.380666,53.592886,-210.000000,209.000000
points_diff,14.592960,265.280149,-1757.000000,1596.000000
rank_trend_diff,0.103636,3.857311,-36.000000,37.000000
form5_winrate_diff,0.012546,0.335842,-1.000000,1.000000
form10_winrate_diff,0.013496,0.260206,-0.900000,1.000000
goal_avg_diff,0.082273,1.770795,-13.200000,14.800000
career_winrate_diff,0.009964,0.153750,-0.696858,0.777778
experience_diff,22.471210,266.365843,-997.000000,997.000000
home_rank,76.804981,52.306435,1.000000,211.000000
away_rank,80.185647,52.916056,1.000000,211.000000


## Étape 5 — Entraînement et comparaison de 3 modèles ML

Split **chronologique** (85 % / 15 %) : plus réaliste que le split aléatoire pour une série temporelle sportive (on entraîne sur le passé, on teste sur les matchs les plus récents).

Modèles comparés :
1. **Régression logistique** (linéaire, interprétable, facilement portable pour la plateforme)
2. **Random Forest**
3. **XGBoost**


In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report
from xgboost import XGBClassifier

RANDOM_STATE = 42
df_sorted = df.sort_values('date')
cut = int(len(df_sorted) * 0.85)
train_idx, test_idx = df_sorted.index[:cut], df_sorted.index[cut:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print(f'Train : {X_train.shape[0]} matchs (jusqu\'au {df_sorted.loc[train_idx, "date"].max().date()})')
print(f'Test  : {X_test.shape[0]} matchs (à partir du {df_sorted.loc[test_idx, "date"].min().date()})')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

Train : 23309 matchs (jusqu'au 2022-01-27)
Test  : 4114 matchs (à partir du 2022-01-27)


In [10]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1),
}

results_summary, fitted_models = [], {}
for name, model in models.items():
    Xtr, Xte = (X_train_sc, X_test_sc) if name == 'Logistic Regression' else (X_train, X_test)
    model.fit(Xtr, y_train)
    pred, proba = model.predict(Xte), model.predict_proba(Xte)
    cv = cross_val_score(model, Xtr, y_train, cv=5, scoring='accuracy')
    fitted_models[name] = model
    results_summary.append({
        'model': name, 'cv_accuracy': cv.mean(), 'test_accuracy': accuracy_score(y_test, pred),
        'test_f1_macro': f1_score(y_test, pred, average='macro'), 'test_log_loss': log_loss(y_test, proba),
    })
    print(f'--- {name} ---')
    print(classification_report(y_test, pred, target_names=['Away win', 'Draw', 'Home win']))

summary_df = pd.DataFrame(results_summary)
summary_df

--- Logistic Regression ---
              precision    recall  f1-score   support

    Away win       0.53      0.64      0.58      1190
        Draw       0.33      0.00      0.00       974
    Home win       0.62      0.85      0.72      1950

    accuracy                           0.59      4114
   macro avg       0.50      0.50      0.43      4114
weighted avg       0.53      0.59      0.51      4114



--- Random Forest ---
              precision    recall  f1-score   support

    Away win       0.55      0.62      0.58      1190
        Draw       0.00      0.00      0.00       974
    Home win       0.61      0.87      0.71      1950

    accuracy                           0.59      4114
   macro avg       0.39      0.49      0.43      4114
weighted avg       0.45      0.59      0.51      4114



--- XGBoost ---
              precision    recall  f1-score   support

    Away win       0.54      0.62      0.58      1190
        Draw       0.33      0.02      0.04       974
    Home win       0.62      0.85      0.71      1950

    accuracy                           0.59      4114
   macro avg       0.50      0.50      0.45      4114
weighted avg       0.53      0.59      0.52      4114



,model,cv_accuracy,test_accuracy,test_f1_macro,test_log_loss
0,Logistic Regression,0.575358,0.588965,0.433927,0.891116
1,Random Forest,0.573899,0.589451,0.432463,0.894518
2,XGBoost,0.567636,0.587749,0.445168,0.897238


### Sélection du meilleur modèle

Les 3 modèles sont proches en accuracy (~0.59, contre une baseline "classe majoritaire" à ~0.47). Le football international reste intrinsèquement bruité — les matchs nuls, en particulier, sont difficiles à capturer pour tous les modèles. On départage donc avec le **F1-macro** (plus adapté à un problème 3 classes déséquilibré) et le **log-loss** (qualité des probabilités affichées par la plateforme), en plus de l'accuracy.


In [11]:
summary_df['rank_score'] = (
    summary_df['test_f1_macro'].rank(ascending=False)
    + summary_df['test_log_loss'].rank(ascending=True)
    + summary_df['test_accuracy'].rank(ascending=False)
)
summary_df = summary_df.sort_values('rank_score')
best_name = summary_df.iloc[0]['model']
print('Modèle retenu :', best_name)
summary_df

Modèle retenu : Logistic Regression


,model,cv_accuracy,test_accuracy,test_f1_macro,test_log_loss,rank_score
0,Logistic Regression,0.575358,0.588965,0.433927,0.891116,5.0
1,Random Forest,0.573899,0.589451,0.432463,0.894518,6.0
2,XGBoost,0.567636,0.587749,0.445168,0.897238,7.0


**Modèle retenu : régression logistique.** Elle obtient le meilleur compromis accuracy / F1-macro / log-loss, et présente l'avantage d'être un modèle linéaire facilement portable (coefficients exportables) pour alimenter une plateforme de prédiction interactive en temps réel.

## Étape 6 — Plateforme de prédiction : application aux 9 matchs restants de la CM 2026

On applique le modèle retenu aux 9 matchs de la Coupe du Monde 2026 encore sans score (quarts de finale à la finale). Pour ces matchs futurs, la forme des équipes est recalculée à partir de **tout leur historique jusqu'à leur dernier match joué** (et non "shiftée", puisqu'il n'y a pas de match courant à exclure).


In [12]:
best_model = fitted_models[best_name]

g2 = long_df.sort_values(['team', 'date']).groupby('team', group_keys=False)
long_df['cur_form5_winrate'] = g2['win'].apply(lambda s: s.rolling(5, min_periods=1).mean())
long_df['cur_form5_avg_gf'] = g2['gf'].apply(lambda s: s.rolling(5, min_periods=1).mean())
long_df['cur_form5_avg_ga'] = g2['ga'].apply(lambda s: s.rolling(5, min_periods=1).mean())
long_df['cur_form10_winrate'] = g2['win'].apply(lambda s: s.rolling(10, min_periods=1).mean())
long_df['cur_career_matches'] = g2.cumcount() + 1
long_df['cur_career_winrate'] = g2['win'].apply(lambda s: s.expanding(min_periods=1).mean())

CUR_COLS = ['cur_form5_winrate', 'cur_form5_avg_gf', 'cur_form5_avg_ga', 'cur_form10_winrate', 'cur_career_matches', 'cur_career_winrate']
latest_snapshot = long_df.sort_values(['team', 'date']).groupby('team').tail(1).set_index('team')[CUR_COLS]

def latest_form(team):
    return latest_snapshot.loc[team] if team in latest_snapshot.index else pd.Series({c: np.nan for c in CUR_COLS})

dfp = to_predict_full.copy()
home_form = dfp['home_team_std'].apply(latest_form).add_prefix('home_')
away_form = dfp['away_team_std'].apply(latest_form).add_prefix('away_')
dfp = pd.concat([dfp.reset_index(drop=True), home_form.reset_index(drop=True), away_form.reset_index(drop=True)], axis=1)

dfp['rank_diff'] = dfp['away_rank'] - dfp['home_rank']
dfp['points_diff'] = dfp['home_total_points'] - dfp['away_total_points']
dfp['rank_trend_diff'] = dfp['home_rank_trend3'] - dfp['away_rank_trend3']
dfp['form5_winrate_diff'] = dfp['home_cur_form5_winrate'] - dfp['away_cur_form5_winrate']
dfp['form10_winrate_diff'] = dfp['home_cur_form10_winrate'] - dfp['away_cur_form10_winrate']
dfp['goal_avg_diff'] = (dfp['home_cur_form5_avg_gf'] - dfp['home_cur_form5_avg_ga']) - (dfp['away_cur_form5_avg_gf'] - dfp['away_cur_form5_avg_ga'])
dfp['career_winrate_diff'] = dfp['home_cur_career_winrate'] - dfp['away_cur_career_winrate']
dfp['experience_diff'] = dfp['home_cur_career_matches'] - dfp['away_cur_career_matches']
dfp['home_form5_winrate'] = dfp['home_cur_form5_winrate']
dfp['away_form5_winrate'] = dfp['away_cur_form5_winrate']
dfp['is_neutral'] = dfp['neutral'].astype(bool).astype(int)
dfp['is_world_cup'], dfp['is_qualifier'], dfp['is_friendly'] = 1, 0, 0

X_new = dfp[FEATURES].fillna(dfp[FEATURES].median())
X_new_in = scaler.transform(X_new) if best_name == 'Logistic Regression' else X_new

proba = best_model.predict_proba(X_new_in)
label_map = {0: 'Victoire extérieure', 1: 'Match nul', 2: 'Victoire à domicile'}
dfp['pred_label'] = [label_map[p] for p in best_model.predict(X_new_in)]
dfp['proba_home_win'] = proba[:, 2].round(3)
dfp['proba_draw'] = proba[:, 1].round(3)
dfp['proba_away_win'] = proba[:, 0].round(3)

dfp[['date', 'home_team', 'away_team', 'home_rank', 'away_rank', 'proba_home_win', 'proba_draw', 'proba_away_win', 'pred_label']]

,date,home_team,away_team,home_rank,away_rank,proba_home_win,proba_draw,proba_away_win,pred_label
0,2026-07-03,Australia,Egypt,23.0,36.0,0.375,0.249,0.376,Victoire extérieure
1,2026-07-03,Argentina,Cape Verde,1.0,65.0,0.791,0.145,0.064,Victoire à domicile
2,2026-07-03,Colombia,Ghana,12.0,64.0,0.563,0.224,0.213,Victoire à domicile
3,2026-07-04,Canada,Morocco,48.0,12.0,0.207,0.230,0.563,Victoire extérieure
4,2026-07-04,Paraguay,France,58.0,2.0,0.099,0.189,0.711,Victoire extérieure
5,2026-07-05,Brazil,Norway,4.0,46.0,0.681,0.196,0.124,Victoire à domicile
6,2026-07-05,Mexico,England,15.0,5.0,0.400,0.270,0.329,Victoire à domicile
7,2026-07-06,Portugal,Spain,6.0,8.0,0.298,0.259,0.443,Victoire extérieure
8,2026-07-06,United States,Belgium,11.0,3.0,0.368,0.270,0.362,Victoire à domicile


## Étape 6bis — Plateforme de prédiction interactive

Le modèle retenu étant linéaire, ses coefficients (+ la moyenne/écart-type du `StandardScaler`) sont exportés en JSON avec les dernières statistiques connues de chaque équipe (rang, points, forme). Cela permet de construire une **plateforme interactive** (widget web) où l'utilisateur choisit deux équipes et obtient instantanément les probabilités de victoire / nul / défaite, sans dépendre d'un serveur Python — le calcul (produit matriciel + softmax) est reproduit en JavaScript à partir des mêmes coefficients que le modèle scikit-learn.

*(Le fichier `platform_widget.html` fourni avec ce notebook contient cette plateforme ; elle a été présentée en direct dans la conversation.)*

Un script `app.py` (Streamlit) est également fourni pour disposer d'une version Python exécutable localement, s'appuyant directement sur `model_best.joblib`.


## Conclusion

- Les deux bases ont été nettoyées, enrichies (forme d'équipe glissante, tendance de classement FIFA) puis fusionnées avec `merge_asof` sur la clé équipe + date.
- La variable cible (Victoire domicile / Nul / Victoire extérieur) a été construite et modélisée par 3 algorithmes.
- La régression logistique a été retenue (meilleur compromis accuracy/F1/log-loss, portabilité).
- Le modèle a été appliqué aux 9 matchs restants de la Coupe du Monde 2026 et intégré dans une plateforme de prédiction interactive.
- **Limite principale** : les matchs nuls restent difficiles à prédire pour tous les modèles testés (F1 très faible sur cette classe), un phénomène bien documenté dans la littérature de prédiction sportive.
